# Module 08-2 솔루션: Circuit Breaker 패턴 구현

**참조 문서**: `docs/part4-production/08-error-handling-resilience.md` 섹션 2.4

In [ ]:
import sys
sys.path.insert(0, '../..')

import time
import threading
from enum import Enum
from dataclasses import dataclass, field
from common.fake_llm import FakeLLM

print("환경 설정 완료")

In [ ]:
# 실습 1 솔루션: CircuitState Enum 정의

class CircuitState(Enum):
    """Circuit Breaker의 3가지 상태."""
    CLOSED = "closed"       # 정상 - 요청 통과
    OPEN = "open"           # 차단 - 요청 거부
    HALF_OPEN = "half_open" # 테스트 - 제한적 통과


class CircuitOpenError(Exception):
    """Circuit Breaker가 OPEN 상태일 때 발생하는 예외."""
    def __init__(self, name: str, remaining_seconds: float):
        self.name = name
        self.remaining_seconds = remaining_seconds
        super().__init__(
            f"Circuit '{name}'이 OPEN 상태입니다. "
            f"{remaining_seconds:.1f}초 후 재시도하세요."
        )


assert hasattr(CircuitState, 'CLOSED')
assert hasattr(CircuitState, 'OPEN')
assert hasattr(CircuitState, 'HALF_OPEN')
print(f"CircuitState 정의 통과: {[s.value for s in CircuitState]}")
print("실습 1 완료!")

In [ ]:
# 실습 2 솔루션: CircuitBreaker 클래스 구현

@dataclass
class CircuitBreaker:
    """Circuit Breaker 구현."""
    name: str
    failure_threshold: int = 5
    recovery_timeout: float = 60.0
    expected_exceptions: tuple = (Exception,)

    _state: CircuitState = field(default=CircuitState.CLOSED, init=False)
    _failure_count: int = field(default=0, init=False)
    _last_failure_time: float = field(default=0.0, init=False)
    _lock: threading.Lock = field(default_factory=threading.Lock, init=False)

    @property
    def state(self) -> CircuitState:
        """현재 상태를 반환합니다. OPEN -> HALF_OPEN 자동 전환 포함."""
        with self._lock:
            if self._state == CircuitState.OPEN:
                elapsed = time.monotonic() - self._last_failure_time
                if elapsed >= self.recovery_timeout:
                    self._state = CircuitState.HALF_OPEN
                    print(f"Circuit '{self.name}': OPEN -> HALF_OPEN ({elapsed:.1f}초 경과)")
            return self._state

    def _record_success(self) -> None:
        """성공을 기록합니다."""
        with self._lock:
            self._failure_count = 0
            if self._state == CircuitState.HALF_OPEN:
                self._state = CircuitState.CLOSED
                print(f"Circuit '{self.name}': HALF_OPEN -> CLOSED (복구됨)")

    def _record_failure(self) -> None:
        """실패를 기록합니다."""
        with self._lock:
            self._failure_count += 1
            self._last_failure_time = time.monotonic()
            if self._failure_count >= self.failure_threshold:
                self._state = CircuitState.OPEN
                print(f"Circuit '{self.name}': -> OPEN (연속 {self._failure_count}회 실패)")

    def call(self, fn, *args, **kwargs):
        """Circuit Breaker를 통해 함수를 호출합니다."""
        current_state = self.state

        if current_state == CircuitState.OPEN:
            remaining = self.recovery_timeout - (
                time.monotonic() - self._last_failure_time
            )
            raise CircuitOpenError(self.name, max(0, remaining))

        try:
            result = fn(*args, **kwargs)
            self._record_success()
            return result
        except self.expected_exceptions:
            self._record_failure()
            raise


print("CircuitBreaker 클래스 정의 완료")

In [ ]:
# 실습 2 검증
cb = CircuitBreaker(
    name="test-api",
    failure_threshold=3,
    recovery_timeout=60.0,
    expected_exceptions=(RuntimeError,),
)

assert cb.state == CircuitState.CLOSED

for i in range(3):
    try:
        cb.call(lambda: (_ for _ in ()).throw(RuntimeError("실패")))
    except RuntimeError:
        pass

assert cb.state == CircuitState.OPEN
print(f"CLOSED -> OPEN 전환 통과: {cb._failure_count}번 실패 후 OPEN")

try:
    cb.call(lambda: "이 코드는 실행되면 안 됩니다")
    assert False
except CircuitOpenError as e:
    print(f"OPEN 상태에서 즉시 거부 확인")

# HALF_OPEN -> CLOSED 테스트
cb2 = CircuitBreaker("recovery-test", failure_threshold=2, recovery_timeout=0.1)
for _ in range(2):
    try:
        cb2.call(lambda: (_ for _ in ()).throw(RuntimeError("실패")))
    except RuntimeError:
        pass

time.sleep(0.15)
assert cb2.state == CircuitState.HALF_OPEN
result = cb2.call(lambda: "복구 성공")
assert result == "복구 성공"
assert cb2.state == CircuitState.CLOSED
print("HALF_OPEN -> CLOSED 복구 통과")
print("실습 2 완료!")

In [ ]:
# 실습 3 솔루션: FakeLLM과 CircuitBreaker 연동
llm = FakeLLM(responses={"분석": "분석 완료", "hello": "안녕하세요"})

llm_cb = CircuitBreaker(
    name="llm-api",
    failure_threshold=3,
    recovery_timeout=0.1,
    expected_exceptions=(ConnectionError,),
)

for i in range(3):
    try:
        llm_cb.call(lambda: (_ for _ in ()).throw(ConnectionError("연결 실패")))
    except ConnectionError:
        print(f"실패 {i+1}회: 연결 오류")

print(f"현재 서킷 상태: {llm_cb.state}")

assert llm_cb.state == CircuitState.OPEN

try:
    llm_cb.call(lambda: llm.invoke("hello").content)
    assert False
except CircuitOpenError as e:
    print(f"서킷 열림 확인: {e}")

print("CircuitBreaker 연동 테스트 통과")
print("실습 3 완료! 모든 실습 완료")